# STEP 4 — Survey Analysis: Subjective Mode Comparison

## Objective

Analyze participant questionnaire responses from the calibration-phase blind gameplay study to establish **subjective mode rankings**.

This notebook:
1. Aggregates responses at the mode level (Mode 1, 2, 3)
2. Computes vote counts for perceived balance categories
3. Computes median ratings for gameplay dimensions
4. Produces a ranked summary of modes based on perceived balance

**Important**: This analysis does NOT merge with gameplay telemetry. Survey data represents subjective player perception, while telemetry represents objective behavior.

---

## Methodology Notes

### Mode Mapping

The survey presented modes as **Mode A, B, C** (blind labels) to avoid bias. These map to the telemetry system's numeric IDs:
* **Mode A → Mode 1**
* **Mode B → Mode 2**
* **Mode C → Mode 3**

*Justification*: The mapping is confirmed by cross-referencing participant usernames with telemetry logs and play order timestamps.

### Likert Scale Transformation

Some survey questions use Likert scales (e.g., "5 (Very fair)"), while others use categorical responses (e.g., "Balanced", "Too many"). We transform these to numeric values for median computation:

* **Numeric ratings**: "5 (Very fair)" → 5
* **Categorical mappings**:
  * "Balanced" → 3
  * "Frequently" → 5, "Occasionally" → 3, "Very rarely" → 1
  * "Too many"/"Too abundant" → 4, "Too few"/"Too limited" → 2

*Justification*: These transformations preserve ordinal relationships while enabling statistical aggregation. We use **median** (not mean) to reduce impact of outliers in small sample sizes.

### Why Median Over Mean?

With only 7 participants, the median is more robust to extreme values and provides a better representation of the "typical" player experience. Mean values can be skewed by a single outlier in such small samples.


## Setup: Load Dependencies

In [1]:
# Core data processing libraries
import pandas as pd
import numpy as np
import os
import json

# Configure pandas display for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## Step 1: Load Survey Data

The survey CSV contains participant responses to the **Mode Comparison Questionnaire**. Each row represents one participant who played all three modes.

In [2]:
# Configuration
DATA_DIR = 'data'
OUTPUT_DIR = os.path.join(DATA_DIR, 'processed')
OUTPUT_SUMMARY = os.path.join(OUTPUT_DIR, 'survey_summary.csv')
OUTPUT_RANKINGS = os.path.join(OUTPUT_DIR, 'survey_rankings.json')

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Auto-detect survey file (handles special characters in filename)
files = [f for f in os.listdir(DATA_DIR) if 'Gameplay' in f and f.endswith('.csv')]
if not files:
    raise FileNotFoundError("Survey CSV file not found in data directory")

SURVEY_FILE = os.path.join(DATA_DIR, files[0])
print(f"Loading survey data from: {files[0]}")

# Load survey data
df_survey = pd.read_csv(SURVEY_FILE)

print(f"\nLoaded {len(df_survey)} participant responses")
print(f"Survey has {len(df_survey.columns)} columns")

# Display first few rows
print("\nSample of survey data:")
display(df_survey.head(3))

,Timestamp,Username,Confirmation of Completion,In which order did you play the modes?,How fair did combat feel in each mode? [Mode A],How fair did combat feel in each mode? [Mode B],How fair did combat feel in each mode? [Mode C],Enemy presence over time felt: [Mode A],Enemy presence over time felt: [Mode B],Enemy presence over time felt: [Mode C],How comfortable was movement and stamina usage? [Mode A],How comfortable was movement and stamina usage? [Mode B],How comfortable was movement and stamina usage? [Mode C],Sprinting and dashing felt: [Mode A],Sprinting and dashing felt: [Mode B],Sprinting and dashing felt: [Mode C],How often did you encounter collectibles? [Mode A],How often did you encounter collectibles? [Mode B],How often did you encounter collectibles? [Mode C],Collectible availability felt: [Mode A],Collectible availability felt: [Mode B],Collectible availability felt: [Mode C],Which mode felt the MOST BALANCED overall?,Which mode felt TOO EASY?,Which mode felt TOO DIFFICULT?,"If you had to play one mode for a longer session, which would you choose?",What did you like MOST about the mode you selected as balanced?,What felt frustrating or unfair in any of the modes?,Research Consent Confirmation
0,2026/01/17 1:46:51 PM GMT+5:30,kash.socialmedia01@gmail.com,"Yes, I played all three modes",A → C → B,5 (Very fair),3,4,Balanced,Too many,Too few,5 (Very comfortable),4,4,Balanced,Too limited,Balanced,Frequently,Frequently,Frequently,Balanced,Balanced,Balanced,Mode B,Mode A,Mode C,Mode C,"Whenever I collect more coins, the game itself...",I want to explore more of the game's map to se...,I confirm that I have read the Research Consen...
1,2026/01/17 5:30:17 PM GMT+5:30,sanilanginige@gmail.com,"Yes, I played all three modes",A → B → C,5 (Very fair),5 (Very fair),5 (Very fair),Balanced,Balanced,Balanced,5 (Very comfortable),4,1 (Very restrictive),Balanced,Balanced,Balanced,Frequently,Frequently,Frequently,Balanced,Balanced,Balanced,Mode B,NaN,NaN,Mode C,Fair health limit and the amount of the enemy,It's very hard to collect the collectables in ...,I confirm that I have read the Research Consen...
2,2026/01/17 8:49:37 PM GMT+5:30,amalshavinda2@gmail.com,"Yes, I played all three modes",A → B → C,4,4,5 (Very fair),Balanced,Balanced,Balanced,4,4,4,Balanced,Balanced,Balanced,Occasionally,Occasionally,Frequently,Balanced,Balanced,Balanced,Mode B,Mode A,Mode C,Mode C,I liked that Mode B felt well balanced between...,"In Mode A, the game felt a bit too easy and di...",I confirm that I have read the Research Consen...


## Step 2: Establish Mode Mapping

Participants played modes labeled as **A, B, C** to maintain blindness. We map these to numeric IDs used in the telemetry system.

**Mapping**: Mode A → 1, Mode B → 2, Mode C → 3

*Verification method*: This mapping was confirmed by comparing participant usernames in the survey with telemetry logs and checking play order timestamps.

In [3]:
# Define mode mapping
MODE_MAPPING = {
    'Mode A': 1,
    'Mode B': 2,
    'Mode C': 3
}

print("Mode Mapping:")
for label, mode_id in MODE_MAPPING.items():
    print(f"  {label} → Mode {mode_id}")

Mode Mapping:
  Mode A → Mode 1
  Mode B → Mode 2
  Mode C → Mode 3


## Step 3: Vote Aggregation

Participants voted on four key perceptions:
1. **Most balanced mode**: Which mode felt fair and well-tuned?
2. **Too easy**: Which mode lacked challenge?
3. **Too difficult**: Which mode was frustrating or unfair?
4. **Would play longer**: Which mode had better long-term engagement potential?

**Counting logic**: For each category, we count how many votes each mode received. Some participants skipped certain questions (NaN values), which we exclude from counts.

In [4]:
# Define vote category columns
vote_categories = {
    'most_balanced': 'Which mode felt the MOST BALANCED overall?',
    'too_easy': 'Which mode felt TOO EASY?',
    'too_difficult': 'Which mode felt TOO DIFFICULT?',
    'would_play_longer': 'If you had to play one mode for a longer session, which would you choose?'
}

vote_results = {}

print("Vote Counts by Category:")
print("=" * 60)

for category, column in vote_categories.items():
    # Count votes for each mode label (A/B/C)
    votes = df_survey[column].value_counts()
    
    # Map to numeric mode IDs and store
    mode_votes = {}
    for mode_label, count in votes.items():
        if pd.notna(mode_label):  # Skip NaN values
            mode_id = MODE_MAPPING.get(mode_label, None)
            if mode_id:
                mode_votes[mode_id] = int(count)
    
    vote_results[category] = mode_votes
    
    # Display results
    print(f"\n{category.replace('_', ' ').title()}:")
    for mode_id in [1, 2, 3]:
        vote_count = mode_votes.get(mode_id, 0)
        print(f"  Mode {mode_id}: {vote_count} votes")

Vote Counts by Category:

Most Balanced:
  Mode 1: 0 votes
  Mode 2: 6 votes
  Mode 3: 1 votes

Too Easy:
  Mode 1: 5 votes
  Mode 2: 1 votes
  Mode 3: 0 votes

Too Difficult:
  Mode 1: 1 votes
  Mode 2: 0 votes
  Mode 3: 4 votes

Would Play Longer:
  Mode 1: 0 votes
  Mode 2: 3 votes
  Mode 3: 4 votes


## Step 4: Rating Computation (Median)

For three gameplay dimensions, participants rated each mode on various aspects:

1. **Combat Fairness**: "How fair did combat feel?"
2. **Exploration Comfort**: "How comfortable was movement and stamina usage?"
3. **Collectible Availability**: "How often did you encounter collectibles?"

**Transformation process**:
- Extract numeric values from Likert scales (e.g., "5 (Very fair)" → 5)
- Map categorical responses to numeric equivalents (e.g., "Balanced" → 3)
- Compute **median** rating per mode (robust to outliers in small samples)

*Why median?* With 7 participants, a single extreme rating can heavily skew the mean. Median better represents the "typical" player experience.

In [5]:
def extract_numeric_rating(value):
    """
    Extract numeric rating from Likert scale text or categorical responses.
    
    Examples:
        "5 (Very fair)" → 5
        "Balanced" → 3
        "Frequently" → 5
        "Too many" → 4
    
    Justification for categorical mappings:
    - "Balanced" is neutral, mapped to midpoint (3)
    - Frequency: Frequently=5 (positive), Occasionally=3 (neutral), Rarely=1 (negative)
    - Extreme responses (Too many/few, Too abundant/limited) mapped to 4/2 to preserve order
    """
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).strip()
    
    # Handle numeric ratings like "5 (Very fair)", "4", etc.
    if value_str[0].isdigit():
        return float(value_str.split()[0])
    
    # Handle categorical ratings with semantic mapping
    rating_map = {
        'Very comfortable': 5,
        'Comfortable': 4,
        'Balanced': 3,
        'Restrictive': 2,
        'Very restrictive': 1,
        'Frequently': 5,
        'Occasionally': 3,
        'Very rarely': 1,
        'Too abundant': 4,
        'Abundant': 4,
        'Too limited': 2,
        'Too many': 4,
        'Too few': 2,
        'Very fair': 5,
        'Fair': 4
    }
    
    # Find matching category and return mapped value
    for text, rating in rating_map.items():
        if text.lower() in value_str.lower():
            return float(rating)
    
    return np.nan

# Define rating dimensions and their corresponding columns
rating_dimensions = {
    'combat_fairness': [
        'How fair did combat feel in each mode? [Mode A]',
        'How fair did combat feel in each mode? [Mode B]',
        'How fair did combat feel in each mode? [Mode C]'
    ],
    'exploration_comfort': [
        'How comfortable was movement and stamina usage? [Mode A]',
        'How comfortable was movement and stamina usage? [Mode B]',
        'How comfortable was movement and stamina usage? [Mode C]'
    ],
    'collectible_availability': [
        'How often did you encounter collectibles? [Mode A]',
        'How often did you encounter collectibles? [Mode B]',
        'How often did you encounter collectibles? [Mode C]'
    ]
}

rating_results = {}

print("\nMedian Ratings by Dimension:")
print("=" * 60)

for dimension, columns in rating_dimensions.items():
    mode_ratings = {}
    
    for idx, col in enumerate(columns, start=1):
        # Extract numeric ratings from text responses
        numeric_values = df_survey[col].apply(extract_numeric_rating)
        
        # Compute median (excluding NaN)
        median_rating = numeric_values.median()
        
        mode_ratings[idx] = median_rating
    
    rating_results[dimension] = mode_ratings
    
    # Display results
    print(f"\n{dimension.replace('_', ' ').title()}:")
    for mode_id, rating in mode_ratings.items():
        print(f"  Mode {mode_id}: {rating:.1f} (median)")


Median Ratings by Dimension:

Combat Fairness:
  Mode 1: 4.0 (median)
  Mode 2: 4.0 (median)
  Mode 3: 4.0 (median)

Exploration Comfort:
  Mode 1: 4.0 (median)
  Mode 2: 4.0 (median)
  Mode 3: 3.0 (median)

Collectible Availability:
  Mode 1: 5.0 (median)
  Mode 2: 3.0 (median)
  Mode 3: 5.0 (median)


## Step 5: Generate Mode-Level Summary Table

Combine vote counts and median ratings into a single summary table for easy comparison across modes.

In [6]:
# Build summary data structure
summary_data = []

for mode_id in [1, 2, 3]:
    row = {
        'modeId': mode_id,
        
        # Vote counts
        'votes_most_balanced': vote_results['most_balanced'].get(mode_id, 0),
        'votes_too_easy': vote_results['too_easy'].get(mode_id, 0),
        'votes_too_difficult': vote_results['too_difficult'].get(mode_id, 0),
        'votes_would_play_longer': vote_results['would_play_longer'].get(mode_id, 0),
        
        # Median ratings
        'median_combat_fairness': rating_results['combat_fairness'].get(mode_id, np.nan),
        'median_exploration_comfort': rating_results['exploration_comfort'].get(mode_id, np.nan),
        'median_collectible_availability': rating_results['collectible_availability'].get(mode_id, np.nan),
    }
    
    summary_data.append(row)

df_summary = pd.DataFrame(summary_data)

print("\nMode-Level Summary Table:")
print("=" * 60)
display(df_summary)

,modeId,votes_most_balanced,votes_too_easy,votes_too_difficult,votes_would_play_longer,median_combat_fairness,median_exploration_comfort,median_collectible_availability
0,1,0,5,1,0,4.0,4.0,5.0
1,2,6,1,0,3,4.0,4.0,3.0
2,3,1,0,4,4,4.0,3.0,5.0


## Step 6: Produce Ranked Summary by Perceived Balance

Rank modes based on "most balanced" votes. This represents the **subjective balance ranking** from player perception.

**Interpretation**:
- **Rank 1**: Most frequently selected as balanced → Best subjective experience
- **Rank 2-3**: Less frequently selected → Perceived as imbalanced

*Note*: This ranking is independent of the objective gameplay telemetry analysis. Alignment or discrepancy between subjective and objective results will be examined in the final integration report (Notebook 06).

In [7]:
# Rank by "most balanced" votes (descending: higher votes = better rank)
df_summary['balance_rank'] = df_summary['votes_most_balanced'].rank(ascending=False, method='min')

# Sort by rank
df_ranked = df_summary.sort_values('balance_rank')

print("\nMode Ranking by Perceived Balance:")
print("=" * 60)
print("(Rank 1 = Most Balanced)\n")
display(df_ranked[['modeId', 'votes_most_balanced', 'balance_rank', 
                    'votes_too_easy', 'votes_too_difficult', 'votes_would_play_longer']])

# Summary interpretation
best_mode = df_ranked.iloc[0]['modeId']
best_votes = df_ranked.iloc[0]['votes_most_balanced']
total_participants = len(df_survey)

print(f"\n→ Mode {int(best_mode)} is ranked #1 with {int(best_votes)}/{total_participants} votes for 'most balanced'")
print(f"→ This represents {100*best_votes/total_participants:.1f}% agreement among participants")


→ Mode 2 is ranked #1 with 6/7 votes for 'most balanced'
→ This represents 85.7% agreement among participants


## Step 7: Save Outputs

Export results for use in downstream analysis:
1. **survey_summary.csv**: Mode-level aggregated data (votes + ratings)
2. **survey_rankings.json**: Structured ranking data with metadata

These outputs will be used in Notebook 06 (Final Integration Report) to compare subjective rankings with objective telemetry-based baseline selection.

In [8]:
# Save summary CSV
df_summary.to_csv(OUTPUT_SUMMARY, index=False)
print(f"✓ Saved mode-level summary: {OUTPUT_SUMMARY}")

# Build rankings JSON
rankings = {
    'ranked_by_balance': df_ranked[['modeId', 'votes_most_balanced', 'balance_rank']].to_dict('records'),
    'vote_counts': vote_results,
    'median_ratings': rating_results,
    'metadata': {
        'total_participants': len(df_survey),
        'timestamp': pd.Timestamp.now().isoformat(),
        'methodology': 'Median ratings, vote counts aggregated by mode'
    }
}

# Save rankings JSON
with open(OUTPUT_RANKINGS, 'w') as f:
    json.dump(rankings, f, indent=2)

print(f"✓ Saved rankings JSON: {OUTPUT_RANKINGS}")

print("\n" + "=" * 60)
print("SURVEY PROCESSING COMPLETE")
print("=" * 60)
print("\nNext step: Proceed to Notebook 05 (Baseline Justification)")

✓ Saved mode-level summary: data\processed\survey_summary.csv
✓ Saved rankings JSON: data\processed\survey_rankings.json

SURVEY PROCESSING COMPLETE

Next step: Proceed to Notebook 05 (Baseline Justification)
